# 4. Generative IA in NLP

## Models and current ecosystem

Not all foundational models are generative and not all generative models are conversational.
> Modelos fundacionales no es el término correcto sino modelos basales xD
![foundational models differences](images/foundational_models_differences.png)

### Important features of a conversational model
- Tokenizer: Dictionary that maps words or subwords to unique integers and vice versa.
- Parameters:
    - Format: Float16, Float32, Float8, etc. It will affect model's size and speed (smaller formats are faster but less precise so less quality). Quantization is the process of converting a model from a higher precision format to a lower precision format to reduce its size and improve inference speed.
    - Number: It will affect model's size and capabilities (better quality with more parameters, but also more computational resources needed).
- Architecture: Transformer-based models are the most common for NLP tasks. We don't need to know the details of the architecture, but it's important to know that different architectures can have different capabilities and performance.
- Context size/window: Number of tokens the model can process at once. A larger context size allows the model to understand and generate longer texts, but also requires more computational resources.
- Training data (e.g., language, domain, etc.): We want a model trained on data that is relevant to our use case to ensure better performance.

> There are studies that show that larger models with more parameters and larger context sizes tend to perform better on a wide range of NLP tasks. However, the quality of the training data and the architecture of the model are also crucial factors that can significantly impact performance. And sometimes smaller models can perform better in specific tasks or domains if they are trained on high-quality, relevant data.

> Also other studies show that after a certain point, increasing the number of parameters or context size yields diminishing returns in performance, because the model may start to overfit the training data or struggle to effectively utilize the additional capacity.

> https://tiktokenizer.vercel.app/

> It's advised to usen the tokenizer associated with the model to ensure compatibility.

https://www.youtube.com/@AndrejKarpathy/videos Amazing videos about transformers and attention mechanisms, even building gpt from scratch.

## First steps with Generative IA in NLP

In [2]:
from transformers import AutoTokenizer
INSTRUCT_MODEL_CON_CASTELLANO = "Qwen/Qwen2.5-0.5B-Instruct" # ~2.4 GB
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL_CON_CASTELLANO) # Tokenizer associated to the model

prompt = "¿Hablas español?"
prompt_tokens = tokenizer.encode(prompt, return_tensors="pt") # shape: (1, n_tokens)
print(f"Sequencia {prompt} to tokens: ")
tokens_as_list = prompt_tokens[0].numpy().tolist()
print(tokens_as_list)

/home/javier/Documents/IA/IA learning/lab7/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sequencia ¿Hablas español? to tokens: 
[30182, 39, 370, 14493, 69888, 30]


In [3]:
vocab = tokenizer.get_vocab()
inv_vocab = {v: k for k, v in vocab.items()}
decoded_tokens = [inv_vocab[token_id] for token_id in tokens_as_list]
print(f"Tokens decoded: {decoded_tokens}")

Tokens decoded: ['Â¿', 'H', 'ab', 'las', 'ĠespaÃ±ol', '?']


> Ġ is used to indicate a space

> Check [Kwen paper](https://arxiv.org/abs/2309.16609) for more details about the model used in this lab.

In [4]:
from transformers import AutoModelForCausalLM
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(INSTRUCT_MODEL_CON_CASTELLANO).to(DEVICE)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1398.16it/s, Materializing param=model.norm.weight]                              


In [5]:
def describe_model(model: AutoModelForCausalLM):
    n_params = model.num_parameters()
    print(f"Total parameters: {n_params:,}")
    print(f"Architecture: {model.config.architectures[0]}")
    try:
        ctx = model.config.max_position_embeddings
    except:
        ctx = model.config.seq_length
    print(f"Maximum context length: {ctx:,} tokens")
    print(f"Parameters dtype: {model.config.torch_dtype}")
    print(f"Vocabulary size: {model.config.vocab_size:,} tokens")

describe_model(model)

`torch_dtype` is deprecated! Use `dtype` instead!


Total parameters: 494,032,768
Architecture: Qwen2ForCausalLM
Maximum context length: 32,768 tokens
Parameters dtype: torch.bfloat16
Vocabulary size: 151,936 tokens


Each model is trained with a prompt pattern, so the model understands the prompt and generates the expected output. For example for a Qwen2.5 Instruct model the pattern is:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
¿Hablas español?<|im_end|>
<|im_start|>assistant
```

## Strategies and parameter tuning

#### Generation strategies
During pre-training the model is trained as a causal language model, predicting the next token given the previous ones (the last layer of the neural network is the probability distribution over the vocabulary), that's why asking the same question can yield different answers. This strategy is Multinomial sampling.
- Multinomial sampling: Randomly selects the next token based on the predicted probabilities. This can lead to more diverse and creative outputs but may also produce less coherent text.

Other strategies are:
- Greedy decoding: Selects the token with the highest probability at each step. This can lead to more coherent text but may also produce repetitive or dull outputs.
- Beam decoding: Maintains multiple candidate sequences (beams) at each step and selects the best one based on the overall probability (Compounded probability). This can balance coherence and diversity but is more computationally expensive.

#### Generation parameters
- Temperature: Controls the randomness of the token selection. A higher temperature (e.g., >1) increases randomness, while a lower temperature (e.g., <1) makes the model more deterministic.
- Top-k sampling: Limits the token selection to the top k most probable tokens. This can help to reduce the chance of selecting low-probability tokens.
- Top-p sampling (nucleus sampling): Limits the token selection to the smallest set of tokens whose cumulative probability exceeds a threshold p. This can help to balance diversity and coherence in the generated text.

> https://medium.com/nlplanet/two-minutes-nlp-most-used-decoding-methods-for-language-models-9d44b2375612

> [transformer explained](https://poloclub.github.io/transformer-explainer/)

> [llm visualization](https://bbycroft.net/llm)

## Practical examples

In [6]:
model.generation_config

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

Generation config of the model shows default parameters for generation. do_sample=True indicates that multinomial sampling is used by default.

In [7]:
gen_kwargs = {
    "max_new_tokens": 50,
}

In [8]:
from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

In [ ]:
import torch
import torch.nn.functional as F

def create_prompt(tokenizer, message):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": message},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return prompt


def generate_response(model, tokenizer, message, streamer=streamer, gen_kwargs=gen_kwargs, return_scores=True, device=DEVICE):
    prompt = create_prompt(tokenizer, message)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device) # return_tensors="pt" for PyTorch
    with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                streamer=streamer,
                return_dict_in_generate=True,
                output_scores=return_scores,
                **gen_kwargs
            )
    return output_ids

def inspect_token_probabilities(response, tokenizer, top_k=5, min_prob=0.01):
    """Show top token probabilities per generated token.

    Args:
        response: model.generate output with output_scores=True
        tokenizer: Hugging Face tokenizer
        top_k: int, max number of top tokens to show
        min_prob: float, minimum probability threshold to display
    """
    scores = response.scores
    sequences = response.sequences[0]
    generated_ids = sequences[-len(scores):]
    special_tokens = tokenizer.all_special_ids

    for token_id, logits in zip(generated_ids, scores):
        if token_id.item() in special_tokens:
            continue  # Skip special tokens
        logits = logits[0] if logits.dim() == 2 else logits
        probs = F.softmax(logits, dim=-1)

        token_str = tokenizer.decode([token_id])
        token_prob = probs[token_id].item() * 100

        top_probs, top_ids = torch.topk(probs, top_k)
        filtered = [(p, idx) for p, idx in zip(top_probs, top_ids) if p.item() >= min_prob]

        print(f"Token: {token_str!r} (generated) | Prob: {token_prob:.2f}%")
        for p, idx in filtered:
            cand = tokenizer.decode([idx.item()])
            print(f"   {cand!r}: {p.item() * 100:.2f}%")
        print()

In [113]:
prompt = "¿Cuál es la capital de Francia?"
response = generate_response(model, tokenizer, prompt, return_scores=True, device=DEVICE)
inspect_token_probabilities(response, tokenizer, top_k=5, min_prob=0.01)

La capital de Francia es París.
Token: 'La' (generated) | Prob: 100.00%
   'La': 100.00%

Token: ' capital' (generated) | Prob: 100.00%
   ' capital': 100.00%

Token: ' de' (generated) | Prob: 100.00%
   ' de': 100.00%

Token: ' Franc' (generated) | Prob: 100.00%
   ' Franc': 100.00%

Token: 'ia' (generated) | Prob: 100.00%
   'ia': 100.00%

Token: ' es' (generated) | Prob: 84.26%
   ' es': 84.26%
   ' actual': 7.87%
   ',': 7.87%

Token: ' Par' (generated) | Prob: 65.80%
   ' Par': 65.80%
   ' N': 10.97%
   ' Paris': 8.91%
   ' Lond': 4.97%
   ' Roma': 4.97%

Token: 'ís' (generated) | Prob: 100.00%
   'ís': 100.00%

Token: '.' (generated) | Prob: 53.22%
   '.': 53.22%
   '.\n\n': 46.78%



In [115]:
prompt = "¿Mejor estación del año para visitar Japón? Responde con una sola palabra!"
response = generate_response(model, tokenizer, prompt, return_scores=True, device=DEVICE)
inspect_token_probabilities(response, tokenizer, top_k=5, min_prob=0.01)

Invierno
Token: 'In' (generated) | Prob: 31.33%
   'In': 31.33%
   'Ver': 24.40%
   'Prim': 17.48%
   'J': 7.60%
   'O': 7.29%

Token: 'vi' (generated) | Prob: 88.93%
   'vi': 88.93%
   'ver': 11.07%

Token: 'erno' (generated) | Prob: 100.00%
   'erno': 100.00%



In [116]:
prompt = "Completa la siguiente frase (sólo dime la respuesta): Caminando por la calle me encontré con"
response = generate_response(model, tokenizer, prompt, gen_kwargs=gen_kwargs, return_scores=True, device=DEVICE)
inspect_token_probabilities(response, tokenizer, top_k=5, min_prob=0.01)

la casa de mi padre, al pie del puente.
Token: 'la' (generated) | Prob: 9.07%
   '"': 15.60%
   'La': 14.35%
   'cam': 13.20%
   'El': 12.15%
   'la': 9.07%

Token: ' casa' (generated) | Prob: 13.41%
   ' casa': 13.41%
   ' pu': 12.86%
   ' luz': 10.02%
   ' gente': 9.61%
   ' calle': 7.23%

Token: ' de' (generated) | Prob: 63.52%
   ' de': 63.52%
   ' del': 10.59%
   '.': 7.76%
   ' que': 6.42%
   ' nueva': 4.23%

Token: ' mi' (generated) | Prob: 48.91%
   ' mi': 48.91%
   ' su': 12.37%
   ' mis': 11.86%
   ' una': 10.04%
   ' un': 10.04%

Token: ' padre' (generated) | Prob: 6.57%
   ' ab': 22.93%
   ' her': 19.41%
   ' amigo': 15.11%
   ' am': 11.29%
   ' t': 9.17%

Token: ',' (generated) | Prob: 18.52%
   '.': 39.07%
   '<|im_end|>': 23.88%
   ',': 18.52%
   '.\n\n': 11.71%
   ' en': 6.81%

Token: ' al' (generated) | Prob: 7.39%
   ' que': 24.76%
   ' un': 12.71%
   ' una': 12.19%
   ' donde': 8.74%
   ' al': 7.39%

Token: ' pie' (generated) | Prob: 4.48%
   ' que': 21.85%
   ' lado

In [117]:
gen_kwargs = {
    "max_new_tokens": 50,
    "temperature": 1.5
}
prompt = "¿Hablas español?"
response = generate_response(model, tokenizer, prompt, gen_kwargs=gen_kwargs, return_scores=True, device=DEVICE)
inspect_token_probabilities(response, tokenizer, top_k=5, min_prob=0.01)

Sí, estoy hablando español. ¿En qué puedo ayudarte hoy?
Token: 'S' (generated) | Prob: 80.23%
   'S': 80.23%
   ' sí': 10.86%
   '¡': 4.92%
   'Si': 3.99%

Token: 'í' (generated) | Prob: 100.00%
   'í': 100.00%

Token: ',' (generated) | Prob: 100.00%
   ',': 100.00%

Token: ' estoy' (generated) | Prob: 23.05%
   ' hab': 29.60%
   ' estoy': 23.05%
   ' soy': 13.98%
   ' puedo': 13.41%
   ' tengo': 5.36%

Token: ' hab' (generated) | Prob: 75.34%
   ' hab': 75.34%
   ' aquí': 10.20%
   ' en': 5.69%
   ' disp': 5.02%
   ' program': 3.75%

Token: 'lando' (generated) | Prob: 100.00%
   'lando': 100.00%

Token: ' español' (generated) | Prob: 23.50%
   ' en': 58.33%
   ' español': 23.50%
   ' de': 18.17%

Token: '.' (generated) | Prob: 87.30%
   '.': 87.30%
   ',': 6.40%
   '.\n\n': 6.30%

Token: ' ¿' (generated) | Prob: 93.75%
   ' ¿': 93.75%
   ' Como': 6.25%

Token: 'En' (generated) | Prob: 55.80%
   'En': 55.80%
   'Cómo': 39.98%
   'Cu': 4.21%

Token: ' qué' (generated) | Prob: 100.00%
  

Increasing temperature will increase randomness of the output. But with smaller models, high temperatures can lead to incoherent outputs.

In [119]:
gen_kwargs = {
    "max_new_tokens": 200,
    "do_sample": False
}
prompt = "¿Cuéntame un chiste corto?"
response = generate_response(model, tokenizer, prompt, gen_kwargs=gen_kwargs, return_scores=True, device=DEVICE)

Claro, aquí tienes uno:

¿Por qué los gatos no usan Facebook?

 porque tienen una cuenta de Instagram.


In [128]:
gen_kwargs = {
    "max_new_tokens": 200,
    "num_beams": 5,
    "do_sample": False
}
prompt = "Escribe un poema sobre la inteligencia artificial en español."
response = generate_response(model, tokenizer, prompt, gen_kwargs=gen_kwargs, return_scores=True, device=DEVICE)
inspect_token_probabilities(response, tokenizer, top_k=5, min_prob=0.01)

Token: 'Aqu' (generated) | Prob: 23.77%
   'Aqu': 23.77%
   'La': 11.23%
   'En': 11.23%
   'Int': 9.31%
   'E': 3.21%

Token: 'í' (generated) | Prob: 99.94%
   'í': 99.94%

Token: ' tienes' (generated) | Prob: 93.04%
   ' tienes': 93.04%
   ' te': 4.63%
   ' está': 1.33%

Token: ' un' (generated) | Prob: 78.79%
   ' un': 78.79%
   ' una': 20.39%

Token: ' po' (generated) | Prob: 97.89%
   ' po': 97.89%

Token: 'ema' (generated) | Prob: 99.95%
   'ema': 99.95%

Token: ' sobre' (generated) | Prob: 69.95%
   ' sobre': 69.95%
   ' en': 17.69%
   ' esc': 1.33%
   ' que': 1.17%

Token: ' la' (generated) | Prob: 98.61%
   ' la': 98.61%

Token: ' intelig' (generated) | Prob: 93.62%
   ' intelig': 93.62%
   ' Intel': 6.02%

Token: 'encia' (generated) | Prob: 99.95%
   'encia': 99.95%

Token: ' artificial' (generated) | Prob: 99.60%
   ' artificial': 99.60%

Token: ' en' (generated) | Prob: 78.04%
   ' en': 78.04%
   ':\n\n': 13.90%
   ' (': 4.51%
   ',': 1.46%

Token: ' español' (generated) | 